In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

df = pd.read_csv("../data/processed/car_price_cleaned.csv")

features = [
    "name", "year", "km_driven", "fuel",
    "seller_type", "transmission", "owner"
]

X = df[features]
y = df["selling_price"]

numeric_features = ["year", "km_driven"]
categorical_features = [
    "name", "fuel", "seller_type", "transmission", "owner"
]

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Dataset shape:", X.shape)
print("Cross-validation folds:", cv.get_n_splits())

Dataset shape: (3577, 7)
Cross-validation folds: 5


In [2]:
models = {
    "Original Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),
    "Log-target Extra Trees": ExtraTreesRegressor(
        n_estimators=250,
        max_depth=25,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

results = []

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    results.append({
        "Model": name,
        "Mean MAE": -scores["test_mae"].mean(),
        "MAE Std": scores["test_mae"].std(),
        "Mean RMSE": -scores["test_rmse"].mean(),
        "Mean R2": scores["test_r2"].mean()
    })

display(pd.DataFrame(results).round(3))

,Model,Mean MAE,MAE Std,Mean RMSE,Mean R2
0,Original Random Forest,174079.350,6184.062,368456.257,0.470
1,Log-target Extra Trees,168375.756,10138.945,369397.904,0.469
